## Data Preprocessing for Machine Learning

The goal of this step is to prepare the raw dataset for machine learning modeling. 
This process involves handling missing values, correcting data types, encoding categorical variables, 
optionally scaling numerical features, and saving the cleaned and transformed dataset for the modeling stage.


### Step 1: Import Required Libraries

In this step, we import the essential Python libraries required for data preprocessing and preparation.

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


### Step 2: Load and Preview the Dataset

In this step, we load the **Telco Customer Churn dataset** and preview the first few rows to understand the dataset structure and key variables.

In [ ]:
# Step 2: Load and preview the dataset
# Load raw dataset
# Assumes the CSV file is available in the working directory.
# In the GitHub repository, the raw dataset is stored under the data/ folder.
df = pd.read_csv("raw_dataset.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 3: Initial Data Inspection

We perform an initial inspection to understand the dataset structure, including the number of rows and columns, data types, and general characteristics.

In [9]:
# Display the number of rows (customers) and columns (features) in the dataset

print(f"Number of customers: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")


Number of customers: 7043
Number of features: 21


In [10]:
# Display dataset structure, data types, and non-null counts

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


**Observations:**

- The dataset contains **7,043** customer records and **21** features.
- Most features are categorical and stored as **object** data types.
- Although `SeniorCitizen` is stored as an **integer**, it represents a **binary categorical** indicator (0 = No, 1 = Yes).
- Continuous numerical features include `tenure` and `MonthlyCharges`.
- The `TotalCharges` feature is stored as an **object** instead of a **numerical** type, 
      which indicates the need for data type correction in the next steps.
- **No missing** values are observed at this stage based on the initial inspection results.

## Step 4: Check for Missing Values

This step identifies missing values across all features to determine which columns require cleaning or transformation.

In [11]:
df.isnull().sum().sort_values(ascending=False)

customerID          0
DeviceProtection    0
TotalCharges        0
MonthlyCharges      0
PaymentMethod       0
PaperlessBilling    0
Contract            0
StreamingMovies     0
StreamingTV         0
TechSupport         0
OnlineBackup        0
gender              0
OnlineSecurity      0
InternetService     0
MultipleLines       0
PhoneService        0
tenure              0
Dependents          0
Partner             0
SeniorCitizen       0
Churn               0
dtype: int64

**Observations:**

- **No missing** values are detected across any of the dataset features at this stage.
- All columns contain complete records at this stage.
- Although no missing values are observed, further preprocessing is still required, particularly for correcting data types such as `TotalCharges`.


## Step 5: Data Type Correction (TotalCharges)

The `TotalCharges` column is stored as an **object dtype** due to non-numeric values (e.g., blank spaces). 
We convert it to a numeric format, coercing invalid values into missing values (NaN).

In [12]:
# Convert TotalCharges to Numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Count missing values introduced after conversion
df['TotalCharges'].isnull().sum()

11

In [13]:
# Verify data type after conversion
df['TotalCharges'].dtype

dtype('float64')

In [14]:
# Check affected records
df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']].head(11)

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,NaN
753,0,20.25,NaN
936,0,80.85,NaN
1082,0,25.75,NaN
1340,0,56.05,NaN
3331,0,19.85,NaN
3826,0,25.35,NaN
4380,0,20.00,NaN
5218,0,19.70,NaN
6670,0,73.35,NaN


**Observations:**

- The `TotalCharges` column was originally stored as an **object dtype** due to non-numeric values (e.g., blank spaces).
- After conversion, **11** values were coerced to **NaN**, indicating previously non-numeric or blank entries.
- Inspection of these records shows they are mostly associated with **zero tenure** (new customers), which explains why `TotalCharges` is missing.
- These missing values will be handled in the next preprocessing step to prepare the feature for modeling.


## Step 6: Handle Missing Values

After conversion, missing values may appear in TotalCharges.
These values are imputed using the **median**, which is more robust to outliers than the mean.

In [15]:
# Imputation by median
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

In [16]:
# Verify missing values after handling
df['TotalCharges'].isnull().sum()

0

**Observations:**

- The missing values introduced in the `TotalCharges` feature were **successfully** imputed using the **median**.
- After imputation, **no missing values** remain in the `TotalCharges` column.
- The **median** is particularly suitable in this case, as the missing values correspond to new customers with zero tenure, preventing the introduction of artificially large values into the dataset.
- The **median** was chosen over the mean to reduce the influence of extreme values and preserve the overall distribution of the feature.
- This step ensures that `TotalCharges` is complete and ready for use in subsequent preprocessing and modeling stages.


## Step 7: Remove Non-Informative Features

The `customerID` column is a unique identifier and does not provide predictive value.
It is removed to prevent noise in the modeling process.

In [17]:
if 'customerID' in df.columns:
    df.drop(columns=['customerID'], inplace=True)


In [18]:
# Validate the dataset structure after removing the customerID column
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


**Observations:**

- The `customerID` feature was **successfully** removed from the dataset.
- As a unique identifier, `customerID` does not contain predictive information relevant to churn prediction.
- Removing this feature reduces noise and dimensionality, helping models focus on meaningful patterns.
- After removal, the dataset contains **20 features**, all of which are relevant for subsequent preprocessing and modeling steps.

**Note**: Apart from `customerID`, all remaining features carry behavioral, contractual, or financial information that may contribute to churn prediction and are therefore retained.



## Step 8: Encode the Target Variable (Churn)

The target variable **Churn** is converted from categorical values (Yes/No) into **numerical** format (1/0) to make it suitable for machine learning algorithms.

In [19]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [20]:
# Verify data type after conversion
df['Churn'].dtype

dtype('int64')

In [21]:
# Ensure that there are no missing values after encoding
df['Churn'].isnull().sum()


0

**Observations:**

- The target variable `Churn` was successfully encoded into a **numerical** format, where **1** represents churned customers and **0** represents non-churned customers.
- This encoding ensures compatibility with machine learning algorithms that require numerical target variables.
- The class distribution indicates that the dataset is **imbalanced**, with fewer churned customers compared to non-churned customers.
- This **imbalance** will be addressed in later stages of the modeling process.

## Step 9: Encode Categorical Features

Categorical features are converted into **numerical** representations using One-Hot Encoding
 
to make the dataset compatible with machine learning models that require numerical inputs.


In [22]:
# Identify categorical (object-type) features that need encoding
categorical_cols = df.select_dtypes(include='object').columns
categorical_cols


Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

In [23]:
# One-Hot Encode categorical features (output as 0/1) and avoid the dummy variable trap
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
df_encoded.head()


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,0,1,0,0,1,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,1,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,0,1,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0


In [24]:
# Check dataset dimensions before and after encoding
print("Original shape:", df.shape)
print("Encoded shape:", df_encoded.shape)
print("New columns added:", df_encoded.shape[1] - df.shape[1])

Original shape: (7043, 20)
Encoded shape: (7043, 31)
New columns added: 11


**Note**: The number of columns increased from 20 to 31 because categorical features were expanded into multiple binary indicator columns through One-Hot Encoding, with one reference category dropped per feature.

In [25]:
# Confirm that no object-type columns remain after encoding
df_encoded.select_dtypes(include='object').columns

Index([], dtype='object')

In [26]:
# Count remaining object-type columns (should be zero)
df_encoded.dtypes.isin(['object']).sum()

0

In [27]:
# Distribution of data types after encoding
df_encoded.dtypes.value_counts()

int32      26
int64       3
float64     2
Name: count, dtype: int64

- **int32**: One-hot encoded categorical features (0/1).
- **int64**: Original integer-based numerical features.
- **float64**: Continuous numerical features such as `MonthlyCharges` and `TotalCharges`.

**Observations:**

- All **categorical (object-type)** features were successfully transformed using One-Hot Encoding.
- The encoded dataset now contains **31 columns**, as categorical features were expanded into multiple binary indicator columns.
- The parameter `drop_first=True` was used to avoid the **dummy variable trap** by removing one reference category per feature.
- All encoded features are stored as **integers (0/1)**, making them suitable for machine learning models.
- This step prepares the dataset for the train-test split and model training stage.

**Note:**
For simplicity, One-Hot Encoding was applied before the train–test split in this notebook. 
 
In the final modeling stage, a `Pipeline` with `OneHotEncoder(handle_unknown="ignore")` will be used to avoid data leakage and handle unseen categories.



## Step 10: Define Features and Target

The dataset is split into input features (X) and the target variable (y) in preparation for model training.

In [28]:
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

print("X shape:", X.shape)
print("y shape:", y.shape)


# Check the dimensions of features and target variable
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Target vector length: {y.shape[0]}")


X shape: (7043, 30)
y shape: (7043,)
Number of samples: 7043
Number of features: 30
Target vector length: 7043


**Observations:**

- The dataset was successfully split into input **features** (**X**) and the **target** variable (**y**).
- **X** contains 30 features after removing the target variable (`Churn`) and applying One-Hot Encoding.
- **y** represents the churn label for each customer.
- Both **X** and **y** contain 7,043 records, ensuring proper alignment between features and target.
- This step prepares the data for the train-test split and model training stages.

## Step 11: Train-Test Split

We split the data into training and testing sets (80/20) to evaluate performance on unseen data. Stratified sampling is used to preserve the original churn class distribution in both sets.

In [29]:
# Split data into train/test sets (80/20) with stratification to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [30]:
# Check train-test split sizes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Percentage split
print(f"Training set size: {len(X_train) / len(X) * 100:.0f}%")
print(f"Testing set size: {len(X_test) / len(X) * 100:.0f}%")


X_train shape: (5634, 30)
X_test shape: (1409, 30)
y_train shape: (5634,)
y_test shape: (1409,)
Training set size: 80%
Testing set size: 20%



X_train contains 5,634 samples and X_test contains 1,409 samples, with 30 features each.

In [31]:
# Verify that class distribution is preserved after stratified split
print("Train class distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest class distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))



Train class distribution (%):
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Test class distribution (%):
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64



Stratified splitting (stratify=y) preserved the churn distribution in both sets (≈ 73% non-churn vs 27% churn).

**Observations:**

- The dataset was successfully split into training and testing sets.
- The **training** set contains **80%** of the data and is used to train the machine learning models.
- The **testing** set contains **20%** of the data and is reserved for evaluating model performance on unseen data.
- Stratified splitting was applied to maintain the original churn class distribution in both sets.
- This split ensures a reliable and fair evaluation of the model on unseen data.

**Note:**  
The 80/20 split is a commonly used practice that provides sufficient data for model learning while reserving enough samples for unbiased evaluation.


## Step 12: Feature Scaling (Optional)

Feature scaling is an optional preprocessing step that may improve model performance for algorithms that are sensitive to feature magnitudes 
(e.g., Logistic Regression, SVM, KNN, Neural Networks).  
However, scaling is not required for tree-based models (e.g., Decision Trees, Random Forest, XGBoost).

At this stage, scaling is not applied yet because the final model has not been selected.  
Instead, we will prepare the feature matrix and identify which columns would be scaled later (typically continuous numeric columns).


In [32]:
# Identify continuous numeric features that are candidates for scaling (usually float columns)
# continuous_cols = X.select_dtypes(include=['float64', 'float32']).columns
# continuous_cols

In [33]:
# print("Number of continuous features to scale later:", len(continuous_cols))
# print("Example columns:", list(continuous_cols)[:10])


In [34]:
# Scaling plan (will be applied during modeling based on the selected algorithm)
# scaling_required_models = ["Logistic Regression", "SVM", "KNN", "Neural Networks"]
# scaling_not_required_models = ["Decision Tree", "Random Forest", "XGBoost"]

# print("Scaling is typically required for:", scaling_required_models)
# print("Scaling is typically not required for:", scaling_not_required_models)


### Observations:
- Feature scaling is kept as an optional step because it depends on the selected machine learning algorithm.
- Continuous numeric columns (e.g., float features) were identified as potential candidates for scaling.
- Scaling will be applied later during the modeling stage only if the chosen model is sensitive to feature magnitudes.


## Step 13: Save Processed Data

Finally, the processed datasets are saved for reuse in the modeling stage. 
Only model-independent data is stored at this stage.


In [35]:
import os

os.makedirs("data/processed", exist_ok=True)

X.to_csv("data/processed/X.csv", index=False)
y.to_csv("data/processed/y.csv", index=False)

X_train.to_csv("data/processed/X_train.csv", index=False)
X_test.to_csv("data/processed/X_test.csv", index=False)
y_train.to_csv("data/processed/y_train.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

#np.save("data/X_train_scaled.npy", X_train_scaled)
#np.save("data/X_test_scaled.npy", X_test_scaled) 

print("Preprocessing completed and files saved successfully.")

Preprocessing completed and files saved successfully.


**Observations:**

- The processed feature matrix (X) and target variable (y) were saved successfully.
- Training and testing datasets were stored separately for use in the modeling stage.
- Only model-independent data was saved at this stage to maintain flexibility for different modeling approaches.


## Conclusion:
In this notebook, the dataset was cleaned, validated, encoded, and split into stratified training and testing sets to ensure a reliable foundation for machine learning workflows.

The preprocessing steps were designed to be fully reproducible, with all transformations applied programmatically rather than relying on manually stored outputs. As a result, the processed datasets are generated locally when needed and are not committed to the repository.

In the next stage, the 3_modeling.ipynb notebook will load the prepared data directly from the preprocessing pipeline to train, compare, and evaluate multiple machine learning models. This stage will also address modeling-specific considerations such as class imbalance handling, feature scaling where required, and the use of pipelines to ensure robustness and prevent data leakage.
